# LLM-as-Judge and Faithfulness Checks

Local metrics are useful, but they do not fully capture whether generated answers follow policy evidence. This notebook adds optional stronger-model judging plus transparent local source-faithfulness checks.

Faithfulness asks whether claims are supported by the supplied evidence. Relevance asks whether the answer addresses the task. Coherence asks whether it is readable and internally consistent. A response can score well on one dimension and fail another, so the final decision keeps them separate.


```mermaid
flowchart TD
    A["Generated answer"] --> B["Local claim checks"]
    A --> C["LLM-as-judge prompt"]
    D["Reference material"] --> B
    D --> C
    B --> E["Decision report"]
    C --> E
```

[LLM-as-judge](https://arxiv.org/abs/2306.05685) means using a stronger language model to score another model's output. It is useful for open-ended answers, but it must be constrained by a clear rubric and source evidence. Judge scores are measurements with their own failure modes: rubric sensitivity, model bias, inconsistent calibration, and preference for verbose or familiar answers.


**Evaluate-on-Hub workflow.** Source: [Hugging Face documentation images](https://huggingface.co/datasets/huggingface/documentation-images/blob/main/evaluate-docs/evaluate-on-hub-banner.png). Evaluation results should be stored and reviewed as artifacts rather than treated as console-only side effects.

<img src="assets/hf-evaluate-on-hub-banner.png" alt="Evaluation workflow banner" style="max-width: 700px; width: 100%; height: auto;">


In [ ]:
from pathlib import Path
import json
import os

from dotenv import load_dotenv
from typing import Literal

from openai import OpenAI, OpenAIError
from pydantic import BaseModel, Field, ValidationError

load_dotenv(dotenv_path=Path(".env"))

EVALUATION_CASES = json.loads(Path("data/evaluation_cases.json").read_text())
REFERENCE_MATERIAL = json.loads(Path("data/reference_material.json").read_text())
REFERENCE_BY_ID = {item["id"]: item for item in REFERENCE_MATERIAL}

## Example Candidate Answers

In a full run, you would import generated answers from notebook 04. To keep this notebook runnable on its own, we define one faithful answer, one hallucinated answer, and one incomplete answer.


In [ ]:
candidate_answers: list[dict[str, object]] = [
    {
        "case_id": "eval-password",
        "model_name": "candidate-faithful",
        "answer": "Do not send passwords to support. Share the account email and use a secure recovery link instead.",
    },
    {
        "case_id": "eval-refund-ticket",
        "model_name": "candidate-hallucinated",
        "answer": "We can always refund digital workshop tickets up to the start time as a courtesy exception.",
    },
    {
        "case_id": "eval-shipping",
        "model_name": "candidate-incomplete",
        "answer": "Please send the order id and I will check what can be done.",
    },
]

candidate_answers

### Exercise: Attach Case Data

Map evaluation case ids to case dictionaries.


In [ ]:
# TODO: create a lookup dictionary by case id.


In [ ]:
# Solution
case_by_id = {case["id"]: case for case in EVALUATION_CASES}
case_by_id["eval-password"]

## Local Faithfulness Checks

A simple local checker cannot understand every claim, but it can catch known policy contradictions. The rules below are intentionally transparent so reviewers can inspect what is being checked.

This is a guardrail, not a general fact checker. Phrase matching can produce false positives when a keyword appears in an incorrect claim and false negatives when a correct answer uses an unexpected paraphrase. Production checks need broader test cases, semantic methods, or human review for high-risk decisions.


In [ ]:
claim_rules = {
    "eval-password": {
        "required_any": ["do not", "secure", "recovery", "account email"],
        "forbidden_any": ["send your password", "one-time passcode", "full card"],
    },
    "eval-refund-ticket": {
        "required_any": ["24 hours", "outside", "not refundable"],
        "forbidden_any": [
            "always refund",
            "up to the start time",
            "courtesy exception",
        ],
    },
    "eval-shipping": {
        "required_any": ["before the order is packed", "cannot", "packed"],
        "forbidden_any": ["always change", "after it is packed"],
    },
    "eval-suspicious-login": {
        "required_any": ["reset", "multi-factor", "password"],
        "forbidden_any": ["send passcode", "share password"],
    },
}

In [ ]:
def local_faithfulness_scan(case_id: str, answer: str) -> dict[str, object]:
    # Check required and forbidden phrase fragments for one generated answer.
    rules = claim_rules[case_id]
    normalized = answer.lower()
    required_hits = [phrase for phrase in rules["required_any"] if phrase in normalized]
    forbidden_hits = [
        phrase for phrase in rules["forbidden_any"] if phrase in normalized
    ]
    return {
        "required_hits": required_hits,
        "forbidden_hits": forbidden_hits,
        "passes_local_scan": bool(required_hits) and not forbidden_hits,
    }


for candidate in candidate_answers:
    # Attach transparent local evidence before any provider call.
    scan_result = local_faithfulness_scan(
        str(candidate["case_id"]),
        str(candidate["answer"]),
    )
    candidate.update(scan_result)

candidate_answers

### Exercise: Identify Blocked Candidates

Return candidate names that fail the local faithfulness scan.


In [ ]:
# TODO: list failing candidate model names.


In [ ]:
# Solution
failing_candidates = [
    candidate["model_name"]
    for candidate in candidate_answers
    if not candidate["passes_local_scan"]
]
failing_candidates

### Exercise: Spot a False Positive

Could the sentence “A secure password may be sent to support” pass the current password rule merely because it contains `secure`?


### Solution

Yes. The current rule accepts any required phrase and only blocks its listed forbidden fragments. This example shows why transparent keyword checks are useful as narrow alarms but unsafe as the only evaluator.


## Provider Configuration

The judge uses [Groq OpenAI compatibility](https://console.groq.com/docs/openai) first and [Mistral chat completions](https://docs.mistral.ai/studio-api/conversations/chat-completion) as the fallback provider. Both use the [OpenAI Python client](https://github.com/openai/openai-python) with provider-specific `base_url` values.

`LLM_PROVIDER` chooses the first provider to try. If both keys are available, the other provider remains in the fallback list. If neither key exists, the notebook uses deterministic dry mode.


In [ ]:
class ProviderConfig(BaseModel):
    # Runtime configuration for one OpenAI-compatible provider.
    name: str
    api_key: str
    base_url: str
    model: str

The next helper reads environment variables, preserves the requested provider order, and discards providers without keys. It returns a list rather than one configuration so runtime fallback is possible.


In [ ]:
PROVIDER_SETTINGS: list[tuple[str, str | None, str, str]] = [
    (
        "groq",
        os.getenv("GROQ_API_KEY"),
        "https://api.groq.com/openai/v1",
        os.getenv("GROQ_MODEL", "llama-3.3-70b-versatile"),
    ),
    (
        "mistral",
        os.getenv("MISTRAL_API_KEY"),
        "https://api.mistral.ai/v1",
        os.getenv("MISTRAL_MODEL", "mistral-large-latest"),
    ),
]

The routing helper now orders those settings and converts only keyed entries into validated provider configurations.


In [ ]:
def configured_providers() -> list[ProviderConfig]:
    # Put the requested provider first, then keep any keyed alternative.
    requested = os.getenv("LLM_PROVIDER", "groq").lower()
    first = requested if requested in {"groq", "mistral"} else "groq"
    ordered_settings = sorted(
        PROVIDER_SETTINGS,
        key=lambda settings: settings[0] != first,
    )

    configs: list[ProviderConfig] = []
    for name, api_key, base_url, model in ordered_settings:
        if api_key:
            configs.append(
                ProviderConfig(
                    name=name,
                    api_key=api_key,
                    base_url=base_url,
                    model=model,
                )
            )
    return configs


PROVIDER_CONFIGS = configured_providers()
[(config.name, config.model) for config in PROVIDER_CONFIGS]

## Structured Judge Schema

A judge score is only useful if it follows a predictable schema. We keep every score on a 1 to 5 scale and restrict the verdict to `pass` or `review`. [Pydantic](https://docs.pydantic.dev/latest/) rejects missing fields, out-of-range scores, and unexpected verdict labels before they enter the decision report.


In [ ]:
class JudgeScore(BaseModel):
    # Structured rubric output from the judge model or dry-mode fallback.
    relevance: int = Field(ge=1, le=5)
    faithfulness: int = Field(ge=1, le=5)
    coherence: int = Field(ge=1, le=5)
    verdict: Literal["pass", "review"]
    rationale: str

In [ ]:
def build_judge_prompt(candidate: dict[str, object]) -> str:
    # Ground the judge in the held-out case and its reference material.
    case = case_by_id[str(candidate["case_id"])]
    references = "\n".join(
        f"- {REFERENCE_BY_ID[reference_id]['title']}: {REFERENCE_BY_ID[reference_id]['text']}"
        for reference_id in case["reference_ids"]
    )
    return (
        "Score the candidate support answer from 1 to 5 for relevance, faithfulness, and coherence.\n"
        "Use only the reference material as ground truth.\n"
        "Return JSON with keys: relevance, faithfulness, coherence, verdict, rationale.\n\n"
        f"Instruction: {case['instruction']}\n\n"
        f"Reference material:\n{references}\n\n"
        f"Ideal answer: {case['ideal_answer']}\n\n"
        f"Candidate answer: {candidate['answer']}"
    )

### Exercise: Preview the Judge Prompt

Build the judge prompt for the first candidate answer.


In [ ]:
# TODO: call build_judge_prompt for candidate_answers[0].


In [ ]:
# Solution
print(build_judge_prompt(candidate_answers[0]))

## Dry Mode and Live Calls

Dry mode is deterministic and conservative: forbidden claims reduce faithfulness, missing required claims reduce relevance, and coherence stays high for readable short text. Live mode asks the configured provider for strict JSON.

Provider errors and invalid structured responses are reported before trying the next configured provider. If every live path fails, the notebook falls back to the transparent local score so the learning flow remains runnable.


In [ ]:
def dry_judge(candidate: dict[str, object]) -> JudgeScore:
    # Score without external API calls by reusing transparent local checks.
    passes = bool(candidate["passes_local_scan"])
    has_forbidden = bool(candidate["forbidden_hits"])
    return JudgeScore(
        relevance=4 if passes else 2,
        faithfulness=1 if has_forbidden else 4,
        coherence=4,
        verdict="pass" if passes else "review",
        rationale="Dry-mode score based on transparent local claim checks.",
    )

Live mode sends the same grounded rubric to either provider and validates the returned JSON immediately. Keeping the network call separate makes it clear which behavior depends on an external service.


In [ ]:
def live_judge(candidate: dict[str, object], config: ProviderConfig) -> JudgeScore:
    # Ask the configured provider for a structured judge score.
    client = OpenAI(api_key=config.api_key, base_url=config.base_url)
    response = client.chat.completions.create(
        model=config.model,
        messages=[
            {"role": "system", "content": "You are a strict evaluation judge."},
            {"role": "user", "content": build_judge_prompt(candidate)},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )
    content = response.choices[0].message.content or "{}"
    return JudgeScore.model_validate_json(content)

Routing is the final responsibility: try each configured provider, report recoverable API or validation failures, and use dry mode only when no live path succeeds.


In [ ]:
def judge_candidate(candidate: dict[str, object]) -> JudgeScore:
    # Try configured providers in order and report failures before dry fallback.
    for config in PROVIDER_CONFIGS:
        try:
            return live_judge(candidate, config)
        except (OpenAIError, ValidationError) as error:
            print(
                f"{config.name} judge failed with {type(error).__name__}; "
                "trying the next available path."
            )
    return dry_judge(candidate)

In [ ]:
for candidate in candidate_answers:
    # Store validated scores beside the answer for later reporting.
    candidate["judge_score"] = judge_candidate(candidate).model_dump()

candidate_answers

### Exercise: Filter Manual Reviews

Return candidates whose judge verdict is not `pass`.


In [ ]:
# TODO: filter candidates that need review.


In [ ]:
# Solution
def require_dict(value: object) -> dict[str, object]:
    # Keep dynamic notebook dictionaries explicit enough for runtime and type checks.
    if isinstance(value, dict):
        return {str(key): item for key, item in value.items()}
    raise TypeError(f"Expected a dictionary, got {type(value).__name__}")


needs_review = []
for candidate in candidate_answers:
    score = require_dict(candidate["judge_score"])
    if score["verdict"] != "pass":
        needs_review.append(candidate)

[
    (
        str(candidate["model_name"]),
        str(require_dict(candidate["judge_score"])["rationale"]),
    )
    for candidate in needs_review
]

## Decision Report

A useful evaluation report does not only declare a winner; it explains failure patterns and what should change next.


In [ ]:
def as_int(value: object) -> int:
    # Judge scores must be integer-like after Pydantic validation.
    if isinstance(value, int):
        return value
    raise TypeError(f"Expected an integer score, got {type(value).__name__}")


def decision_label(candidate: dict[str, object]) -> str:
    # Block unfaithful answers, review incomplete answers, accept clean ones.
    score = require_dict(candidate["judge_score"])
    faithfulness = as_int(score["faithfulness"])
    relevance = as_int(score["relevance"])
    has_forbidden_hits = bool(candidate["forbidden_hits"])
    has_required_hits = bool(candidate["required_hits"])

    if has_forbidden_hits or faithfulness <= 2:
        return "block: likely unfaithful"
    if not has_required_hits or relevance <= 2:
        return "review: incomplete"
    return "accept"


report_rows = [
    {
        "case_id": str(candidate["case_id"]),
        "model_name": str(candidate["model_name"]),
        "decision": decision_label(candidate),
        "required_hits": candidate["required_hits"],
        "forbidden_hits": candidate["forbidden_hits"],
        "judge_score": candidate["judge_score"],
    }
    for candidate in candidate_answers
]
report_rows

### Exercise: Name the Next Data Fix

If a candidate is incomplete but not forbidden, what should you add to the next training set?


### Solution

Add targeted examples that show the missing required claim in context, then keep this case or a similar one in the held-out evaluation set to verify generalization.


## Further Reading

- [Judging LLM-as-a-Judge](https://arxiv.org/abs/2306.05685)
- [G-Eval](https://arxiv.org/abs/2303.16634)
- [Groq structured outputs](https://console.groq.com/docs/structured-outputs)
- [Mistral structured outputs](https://docs.mistral.ai/studio-api/conversations/structured-output)
- [OpenAI Python client](https://github.com/openai/openai-python)
